In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("init_load_flag", "0")
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

**Data Reading From Source**

In [0]:
df = spark.sql('select * from databricks_catalog.silver.customers_silver')

**Removing Duplicates**

In [0]:
df = df.dropDuplicates(subset=['customer_id'])

#**Dividing New vs Old Records**

In [0]:
if init_load_flag == 0:
    df_old = spark.sql('''select DimCustomerKey, customer_id, create_date, update_date from databricks_catalog.gold.DimCustomers''')

else:
    df_old = spark.sql('''select 0 DimCustomerKey, customer_id, 0 create_date, 0 update_date from databricks_catalog.silver.customers_silver where 1 = 0''')

In [0]:
df_old.display()

**Renaming Columns of df_old**

In [0]:
df_old = df_old.withColumnRenamed('DimCustomerKey',"old_DimCustomerKey")\
               .withColumnRenamed("customer_id","old_customer_id")\
               .withColumnRenamed("create_date","old_create_date")\
               .withColumnRenamed("update_date","old_update_date")
               

**Applying Join with the Old Records**

In [0]:
df_join = df.join(df_old, df.customer_id == df_old.old_customer_id, 'left')

In [0]:
df_join.display()

**Separating New vs Old Records**

In [0]:
df_new = df_join.filter(df_join['old_DimCustomerKey'].isNull())

In [0]:
df_old = df_join.filter(df_join['old_DimCustomerKey'].isNotNull())

**Preparing df_old**

In [0]:
#Dropping all the columns that are not required

df_old = df_old.drop('old_customer_id','old_update_date')

# Renaming "old_DimCustomerKey" to "DimCustomerKey"
df_old = df_old.withColumnRenamed("old_DimCustomerKey","DimCustomerKey")

# Renaming "old_create_date" column to "create_date"
df_old = df_old.withColumnRenamed("old_create_date","create_date")
df_old = df_old.withColumn("create_date",to_timestamp(col("create_date")))

# Renaming "update_date" column with current timestamp
df_old = df_old.withColumn("update_date", current_timestamp())

In [0]:
df_old.display()

**Preparing df_new**

In [0]:
#Dropping all the columns that are not required

df_new = df_new.drop('old_DimCustomerKey','old_customer_id','old_update_date','old_create_date')

# Recreating "update_date", "current_date" columns with current timestamp
df_new = df_new.withColumn("update_date", current_timestamp())
df_new = df_new.withColumn("create_date", current_timestamp())


In [0]:
df_new.display()

**Surrogate Key - From 1**

In [0]:
df_new = df_new.withColumn('DimCustomerKey',monotonically_increasing_id()+lit(1))

In [0]:
df_new.display()

**Adding Max Surrogate Key**

In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0 
else:
    df_maxsurrogate = spark.sql("select max(DimCustomerKey) as max_surrogate_key from databricks_catalog.gold.DimCustomers")
    
    # Converting df_maxsurrogate to max_surrogate_key 
    max_surrogate_key = df_maxsurrogate.collect()[0]['max_surrogate_key']

    

In [0]:
df_new = df_new.withColumn("DimCustomerKey",lit(max_surrogate_key)+col("DimCustomerKey"))

In [0]:
df_new.display()

**Union of df_old and df_new**

In [0]:
df_final = df_new.unionByName(df_old)


In [0]:
df_final.display()

**SCD Type - 1**

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('databricks_catalog.gold.DimCustomers'):
    dlt_obj = DeltaTable.forPath(spark,'abfss://gold@stgdatabricksete14.dfs.core.windows.net/DimCustomers')

    dlt_obj.alias('target').merge(df_final.alias("source"),"target.DimCustomerKey = source.DimCustomerKey")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()
    
else:
    df_final.write.mode("overwrite")\
    .option('path','abfss://gold@stgdatabricksete14.dfs.core.windows.net/DimCustomers')\
    .saveAsTable("databricks_catalog.gold.DimCustomers")

 

    


In [0]:
%sql
select * from databricks_catalog.gold.dimcustomers